# Model 3: MobileNetV2 for Automatic Waste Classification

**Module**: SE4050 – Deep Learning (2026)  
**Author**: S. S. Kumbukage (IT23155534)  
**Track**: Supervised Deep Learning (Waste Classification)  

---

## 1. Architecture Overview & Theoretical Rationale

### What is MobileNetV2?
MobileNetV2 is an efficient, lightweight convolutional neural network architecture introduced by Google (Sandler et al., 2018) specifically designed for mobile and resource-constrained edge computing environments.

### Key Architectural Innovations:
1. **Depthwise Separable Convolutions**:
   Standard convolution jointly computes spatial features and cross-channel correlations. MobileNet factorizes standard convolutions into:
   - **Depthwise Convolution**: Applies a single $3 \times 3$ convolutional filter per input channel (spatial filtering).
   - **Pointwise Convolution**: Applies a $1 \times 1$ convolution to linearly combine the outputs across all channels.
   *Theoretical advantage*: Reduces computational operations (FLOPs) and parameter count by roughly $8$ to $9$ times compared to traditional convolutions:
   $$\frac{\text{Cost}_{\text{Depthwise Separable}}}{\text{Cost}_{\text{Standard}}} = \frac{D_K \cdot D_K \cdot M \cdot D_F \cdot D_F + M \cdot N \cdot D_F \cdot D_F}{D_K \cdot D_K \cdot M \cdot N \cdot D_F \cdot D_F} = \frac{1}{N} + \frac{1}{D_K^2} \approx \frac{1}{9}$$
   (where $D_K = 3$ is the kernel size, $M$ is input channels, $N$ is output channels).

2. **Inverted Residuals**:
   Unlike classic ResNet architectures (which compress channels first, perform convolution, and then expand), MobileNetV2 uses an **inverted bottleneck** structure:
   - **Low-dimensional input** $\to$ **$1 \times 1$ Expansion** (expands channels by factor $t=6$) $	o$ **$3 \times 3$ Depthwise Conv** $	o$ **$1 \times 1$ Projection** back to low dimensions.
   - Shortcut (residual) connections are placed directly between the thin bottlenecks.

3. **Linear Bottlenecks**:
   Nonlinear activation functions (such as ReLU) can inadvertently collapse manifold structures and destroy information in low-dimensional tensor spaces. MobileNetV2 removes non-linearities at the narrow bottleneck projection layer and retains linear activations, significantly preserving feature representations.

---

## 2. Experimental Methodology: Two-Stage Transfer Learning
To achieve high generalization on the 6 waste categories (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) without overfitting, this notebook uses a controlled **Two-Stage Transfer Learning Protocol**:
- **Stage 1 (Feature Extraction)**: The pre-trained ImageNet backbone is frozen (`base_model.trainable = False`). A custom classification head is trained with an initial learning rate of $10^{-3}$ to adapt top-level representations.
- **Stage 2 (Fine-Tuning)**: The top 30 layers of the backbone are unfrozen and trained jointly with the classification head at a conservative learning rate of $10^{-5}$ to refine waste-specific spatial features.


In [1]:
# Core imports and environment setup
from pathlib import Path
import os
import sys
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)

# Verify TensorFlow and hardware acceleration
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Hardware Acceleration: GPU Detected ({gpus[0].name})")
else:
    print("Hardware Acceleration: CPU Mode (Training will proceed on CPU)")


TensorFlow Version: 2.18.0
Hardware Acceleration: CPU Mode (Training will proceed on CPU)


## 3. Reproducibility Configuration & Directory Setup
Reproducibility is a core requirement of the SE4050 marking rubric. We fix seeds across the Python runtime, NumPy, and TensorFlow. We also configure relative project directories dynamically so the notebook runs seamlessly in local environments, Google Colab, or Kaggle.


In [2]:
# 1. Reproducible Seed Configuration
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 2. Dynamic Path Resolution
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_IMAGES_DIR = DATA_DIR / "raw" / "Garbage_Dataset_Classification" / "images"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
TABLES_DIR = RESULTS_DIR / "tables"

for directory in [FIGURES_DIR, METRICS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Processed CSVs:", PROCESSED_DIR)
print("Raw Images   :", RAW_IMAGES_DIR)
print("Results Dir  :", RESULTS_DIR)


Project Root : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification
Processed CSVs: c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification\data\processed
Raw Images   : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification\data\raw\Garbage_Dataset_Classification\images
Results Dir  : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification\results


## 4. Ingesting Preprocessed Splits (Zero Data Leakage)
We load the stratified, leakage-free dataset manifests produced in `01_EDA_Preprocessing.ipynb`:
- `train.csv` (9,692 samples)
- `validation.csv` (2,114 samples)
- `test.csv` (2,091 samples)

The test set remains **strictly unseen** and is loaded only for final unbiased model evaluation.


In [3]:
# Load split manifests
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

# Resolve absolute image paths
train_df["image_path"] = train_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))
val_df["image_path"] = val_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))
test_df["image_path"] = test_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))

# Define class categories and label mappings
CLASSES = sorted(train_df["class"].unique())
NUM_CLASSES = len(CLASSES)
CLASS_TO_LABEL = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}
LABEL_TO_CLASS = {idx: cls_name for idx, cls_name in enumerate(CLASSES)}

print(f"Total Dataset: {len(train_df) + len(val_df) + len(test_df)} images across {NUM_CLASSES} classes:")
for label, class_name in LABEL_TO_CLASS.items():
    print(f"  Class {label}: {class_name:<10} | Train: {(train_df['class'] == class_name).sum():<5} | Val: {(val_df['class'] == class_name).sum():<5} | Test: {(test_df['class'] == class_name).sum():<5}")


Total Dataset: 13897 images across 6 classes:
  Class 0: cardboard  | Train: 1548  | Val: 322   | Test: 344  
  Class 1: glass      | Train: 1751  | Val: 380   | Test: 367  
  Class 2: metal      | Train: 1440  | Val: 326   | Test: 317  
  Class 3: paper      | Train: 1605  | Val: 364   | Test: 346  
  Class 4: plastic    | Train: 1598  | Val: 347   | Test: 342  
  Class 5: trash      | Train: 1750  | Val: 375   | Test: 375  


## 5. Input Data Pipeline & On-the-Fly Augmentation
We construct a high-performance `tf.data.Dataset` streaming pipeline:
- **Resizing**: Uniform spatial dimensions $(224, 224, 3)$.
- **Normalization**: `tf.keras.applications.mobilenet_v2.preprocess_input` scales pixels into $[-1.0, +1.0]$, exactly matching MobileNetV2's pre-training condition.
- **Data Augmentation (Train Set Only)**:
  - Random horizontal reflection (`RandomFlip("horizontal")`)
  - Small rotations ($\pm 10\%$)
  - Subtle scaling ($\pm 10\%$ zoom)
  - Random translation ($\pm 10\%$)
- **Optimization**: `prefetch(tf.data.AUTOTUNE)` overlaps data ingestion on the CPU with model forward/backward passes on the GPU.


In [4]:
# Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Data augmentation layer (active only during training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
], name="data_augmentation")

def load_and_preprocess_image(path, label):
    """Loads, decodes, resizes, and scales an image using MobileNetV2 preprocessing."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def build_dataset(df, is_training=False):
    """Constructs an optimized tf.data.Dataset pipeline."""
    paths = df["image_path"].values
    labels = df["label"].values
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if is_training:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    
    if is_training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
        
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, is_training=True)
val_ds = build_dataset(val_df, is_training=False)
test_ds = build_dataset(test_df, is_training=False)

print(f"Pipelines instantiated: Train batches: {len(train_ds)}, Val batches: {len(val_ds)}, Test batches: {len(test_ds)}")


Pipelines instantiated: Train batches: 303, Val batches: 67, Test batches: 66


## 6. Model Architecture & Custom Classification Head
We instantiate the pre-trained `MobileNetV2` backbone initialized with ImageNet weights (excluding the top 1000-class classifier). 

We append a regularized classification head:
1. `GlobalAveragePooling2D`: Compresses spatial activations ($7 \times 7 \times 1280$) into a single 1280-dimensional feature vector, preventing spatial overfitting.
2. `BatchNormalization`: Stabilizes activation distributions before dense projection.
3. `Dropout(0.3)`: Zeroes $30\%$ of activations at random to enforce redundant representations.
4. `Dense(128, activation='relu')`: Intermediate non-linear mapping.
5. `Dropout(0.2)`: Secondary regularization.
6. `Dense(6, activation='softmax')`: Posterior class probabilities $\hat{y} \in [0, 1]^6$.


In [5]:
def create_mobilenetv2_classifier(num_classes=6):
    """Constructs MobileNetV2 with a tailored transfer learning head."""
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Freeze backbone for Stage 1

    inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = tf.keras.layers.BatchNormalization(name="head_batch_norm")(x)
    x = tf.keras.layers.Dropout(0.3, name="head_dropout_1")(x)
    x = tf.keras.layers.Dense(128, activation='relu', name="head_dense_128")(x)
    x = tf.keras.layers.Dropout(0.2, name="head_dropout_2")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name="waste_predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Waste_MobileNetV2")
    return model, base_model

model, base_model = create_mobilenetv2_classifier(num_classes=NUM_CLASSES)
model.summary()

total_params = model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters (Stage 1): {trainable_params:,}")


KeyboardInterrupt: 

## 7. Stage 1 Training: Feature Extraction (Frozen Backbone)
We compile and train the classification head while the pre-trained ImageNet weights remain frozen.
- **Loss Function**: `SparseCategoricalCrossentropy()` (computes cross-entropy loss directly from integer labels).
- **Optimizer**: `Adam(learning_rate=1e-3)` with default momentum $\beta_1=0.9, \beta_2=0.999$.
- **Callbacks**:
  - `EarlyStopping`: Monitors validation loss with a patience of 4 epochs to prevent overfitting.
  - `ReduceLROnPlateau`: Halves the learning rate if validation loss plateaus for 2 consecutive epochs.


In [ ]:
# Stage 1 Configuration
STAGE1_EPOCHS = 8

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting Stage 1: Feature Extraction...")
start_time_s1 = time.time()

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    callbacks=callbacks_stage1
)

stage1_duration = time.time() - start_time_s1
print(f"Stage 1 Complete. Training Time: {stage1_duration:.2f} seconds.")


## 8. Stage 2 Training: Fine-Tuning (Unfreezing Top 30 Layers)
Now that the custom classification head is trained, we unfreeze the top $30$ layers of the MobileNetV2 backbone.
- We deliberately keep earlier layers frozen because shallow layers encode universal low-level visual features (edges, textures, gradients) that do not need modification.
- We use a **very small learning rate** ($10^{-5}$) to prevent large gradient updates from corrupting the pre-trained weights.


In [ ]:
# Unfreeze top layers of the backbone
base_model.trainable = True

# Freeze all layers except the last 30
FINE_TUNE_LAYERS = 30
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

trainable_params_s2 = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"Backbone unfrozen: Last {FINE_TUNE_LAYERS} layers are trainable.")
print(f"Trainable Parameters (Stage 2): {trainable_params_s2:,}")

# Re-compile with conservative learning rate
STAGE2_EPOCHS = 10
TOTAL_EPOCHS = len(history_stage1.history['loss']) + STAGE2_EPOCHS

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Starting Stage 2: Fine-Tuning...")
start_time_s2 = time.time()

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history_stage1.history['loss']),
    callbacks=callbacks_stage2
)

stage2_duration = time.time() - start_time_s2
print(f"Stage 2 Complete. Fine-tuning Time: {stage2_duration:.2f} seconds.")


## 9. Learning Curves & Training Trajectory
We plot training and validation loss and accuracy across both stages to evaluate convergence, generalization, and the performance boost provided by fine-tuning.


In [ ]:
# Concatenate history from Stage 1 and Stage 2
acc = history_stage1.history['accuracy'] + history_stage2.history['accuracy']
val_acc = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
loss = history_stage1.history['loss'] + history_stage2.history['loss']
val_loss = history_stage1.history['val_loss'] + history_stage2.history['val_loss']

transition_epoch = len(history_stage1.history['loss'])

plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#1f77b4', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#ff7f0e', lw=2)
plt.axvline(x=transition_epoch - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('MobileNetV2 Accuracy Curve', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('Accuracy', fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#1f77b4', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#ff7f0e', lw=2)
plt.axvline(x=transition_epoch - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('MobileNetV2 Loss Curve', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=10)
plt.ylabel('Loss', fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')

plt.tight_layout()
learning_curve_path = FIGURES_DIR / "mobilenetv2_learning_curves.png"
plt.savefig(learning_curve_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Learning curves saved to: {learning_curve_path}")


## 10. Comprehensive Model Evaluation on Unseen Test Data
Now, we evaluate MobileNetV2 on `test.csv` (2,091 images). We compute:
- Overall Test Accuracy
- Per-class Precision, Recall, and F1-Score
- Macro and Weighted F1-Scores
- 6x6 Normalized Confusion Matrix Heatmap
- Inference Latency and Parameter Complexity


In [ ]:
# 1. Run predictions across test set
print("Generating predictions on unseen test data...")
start_infer = time.time()
test_pred_probs = model.predict(test_ds, verbose=1)
total_infer_time = time.time() - start_infer

test_predictions = np.argmax(test_pred_probs, axis=1)
test_true_labels = test_df["label"].values

# 2. Compute quantitative metrics
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    test_true_labels, test_predictions, average='macro'
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    test_true_labels, test_predictions, average='weighted'
)
avg_latency_ms = (total_infer_time / len(test_df)) * 1000

print("\n=================== MOBILENETV2 TEST PERFORMANCE ===================")
print(f"Test Accuracy       : {test_accuracy * 100:.2f}%")
print(f"Macro F1-Score      : {f1_macro:.4f}")
print(f"Weighted F1-Score   : {f1_weighted:.4f}")
print(f"Macro Precision     : {precision_macro:.4f}")
print(f"Macro Recall        : {recall_macro:.4f}")
print(f"Avg Inference Speed : {avg_latency_ms:.2f} ms / image")
print("===================================================================")


## 11. Confusion Matrix & Per-Class Performance
The confusion matrix exposes class-level misclassifications (e.g., distinguishing between visually similar materials such as plastic vs. glass, or paper vs. cardboard).


In [ ]:
# Generate classification report
cls_report = classification_report(
    test_true_labels,
    test_predictions,
    target_names=CLASSES,
    digits=4,
    output_dict=True
)
cls_report_df = pd.DataFrame(cls_report).transpose()
report_csv_path = TABLES_DIR / "mobilenetv2_classification_report.csv"
cls_report_df.to_csv(report_csv_path)
print("Classification report saved to:", report_csv_path)
print(classification_report(test_true_labels, test_predictions, target_names=CLASSES, digits=4))

# Plot normalized confusion matrix
cm = confusion_matrix(test_true_labels, test_predictions)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 7))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2%',
    cmap='Blues',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cbar=True
)
plt.title('MobileNetV2 Normalized Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Waste Category', fontsize=11)
plt.ylabel('True Waste Category', fontsize=11)
plt.tight_layout()

cm_path = FIGURES_DIR / "mobilenetv2_confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion matrix saved to: {cm_path}")


## 12. Qualitative Error Analysis
A rigorous deep learning discussion requires inspecting specific failure modes. We extract misclassified test images to analyze the visual ambiguities causing the errors.


In [ ]:
# Find misclassified sample indices
misclassified_indices = np.where(test_predictions != test_true_labels)[0]
print(f"Total misclassified test images: {len(misclassified_indices)} out of {len(test_df)}")

# Display 6 representative misclassified samples
plt.figure(figsize=(15, 8))
sample_count = min(6, len(misclassified_indices))

for idx, err_idx in enumerate(misclassified_indices[:sample_count]):
    img_path = test_df.iloc[err_idx]["image_path"]
    true_class = LABEL_TO_CLASS[test_true_labels[err_idx]]
    pred_class = LABEL_TO_CLASS[test_predictions[err_idx]]
    conf = test_pred_probs[err_idx][test_predictions[err_idx]] * 100
    
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    
    plt.subplot(2, 3, idx + 1)
    plt.imshow(img)
    plt.title(f"True: {true_class}\nPred: {pred_class} ({conf:.1f}%)", color='red', fontsize=10, fontweight='bold')
    plt.axis('off')

plt.suptitle("MobileNetV2 Qualitative Error Analysis (Misclassified Test Samples)", fontsize=13, fontweight='bold')
plt.tight_layout()

err_analysis_path = FIGURES_DIR / "mobilenetv2_error_analysis.png"
plt.savefig(err_analysis_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Error analysis plot saved to: {err_analysis_path}")


## 13. Exporting Model Checkpoints & Benchmark Metrics
Finally, we export the trained model and serialize benchmark metrics to JSON and CSV for inclusion in the group comparison table.


In [ ]:
# Save the trained model checkpoint
model_save_path = PROJECT_ROOT / "src" / "models" / "mobilenetv2_waste_model.keras"
model.save(model_save_path)
model_size_mb = os.path.getsize(model_save_path) / (1024 * 1024)
print(f"Model saved to: {model_save_path} ({model_size_mb:.2f} MB)")

# Benchmark Summary Dictionary
benchmark_metrics = {
    "model_name": "MobileNetV2",
    "architecture_type": "Inverted Residuals + Depthwise Separable CNN",
    "input_resolution": "224x224x3",
    "total_parameters": int(total_params),
    "trainable_parameters_stage1": int(trainable_params),
    "trainable_parameters_stage2": int(trainable_params_s2),
    "model_size_mb": round(model_size_mb, 2),
    "test_accuracy": round(float(test_accuracy), 4),
    "macro_precision": round(float(precision_macro), 4),
    "macro_recall": round(float(recall_macro), 4),
    "macro_f1": round(float(f1_macro), 4),
    "weighted_f1": round(float(f1_weighted), 4),
    "avg_inference_latency_ms": round(float(avg_latency_ms), 2),
    "training_time_seconds": round(stage1_duration + stage2_duration, 2)
}

# Export metrics JSON
metrics_json_path = METRICS_DIR / "mobilenetv2_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(benchmark_metrics, f, indent=4)
print(f"Metrics JSON saved to: {metrics_json_path}")

# Export tabular summary CSV
summary_df = pd.DataFrame([benchmark_metrics])
summary_csv_path = TABLES_DIR / "mobilenetv2_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"Summary table saved to: {summary_csv_path}")

print("\n--- ALL MOBILENETV2 DELIVERABLES COMPLETED SUCCESSFULLY ---")
